In [ ]:
# ============================================================
# CELL 1: Required libraries install karein
# ============================================================
!pip install -q chromadb sentence-transformers pypdf2 transformers accelerate bitsandbytes python-docx
!pip install -q google-generativeai openai
!pip install -q langchain langchain-community torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2: Saare necessary imports
# ============================================================
import os, re, textwrap, getpass, shutil
import numpy as np
import pandas as pd
from PyPDF2 import PdfReader
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from openai import OpenAI
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from docx import Document
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CELL 4: ChromaDB (vector store) setup – NEW API
# ============================================================
import chromadb

client = chromadb.PersistentClient(path="/content/chroma_db")
collection_name = "rag_documents"

# Agar pehle se collection mojood hai to delete karein (fresh start)
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(collection_name)

collection = client.create_collection(collection_name)
print(f"✅ ChromaDB collection '{collection_name}' ready hai")

✅ ChromaDB collection 'rag_documents' ready hai


In [ ]:
# ============================================================
# CELL 5: PDF se text nikalo aur chunks banao
# ============================================================
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

def split_text(text, chunk_size=500, overlap=50):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i+chunk_size]
        if len(chunk) > 100:
            chunks.append(chunk)
    return chunks

def ingest_document(file_path):
    text = extract_text_from_pdf(file_path)
    chunks = split_text(text)
    if not chunks:
        return 0
    embeddings = embedder.encode(chunks).tolist()
    collection.add(
        embeddings=embeddings,
        documents=chunks,
        ids=[f"doc_{i}" for i in range(len(chunks))]
    )
    return len(chunks)

print("✅ PDF processing functions ready")

✅ PDF processing functions ready


In [ ]:
# ============================================================
# CELL 6: Sawal ke liye similar chunks dhundo (Retrieval)
# ============================================================
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    return results['documents'][0]  # top_k chunks
print("✅ Retrieval function ready")

✅ Retrieval function ready


In [ ]:
# ============================================================
# CELL 7a: Gemini via OpenRouter with Fallback (Rate Limit Fix)
# ============================================================
import time

def setup_gemini():
    api_key = "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"
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )
    return client

def gemini_answer(client, context, question):
    """
    Try multiple free models in sequence.
    If one returns 429 (rate limited), wait and try the next.
    """
    prompt = f"""You are an AI assistant that answers questions strictly based on the provided context.
If the answer is not in the context, say 'Not found in the documents'.

Context:
{context}

Question: {question}
Answer with source citations."""

    # --- Free models ki priority list ---
    models = [
        "google/gemma-4-31b-it:free",            # aapka pasandida
        "google/gemini-2.0-flash-exp:free",       # fast, usually available
        "google/gemini-2.0-flash-lite-preview-02-05:free",
        "google/gemma-2-2b-it:free",               # older but stable
        "google/gemini-1.5-flash:free"             # another fallback
    ]

    for model_name in models:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                print(f"⚠️  Model '{model_name}' rate limited. Waiting 5s...")
                time.sleep(5)               # thoda intezar karein
                continue                     # agla model try karein
            else:
                # Agar koi aur error hai (e.g., model not found)
                print(f"❌ Error with '{model_name}': {err[:120]}")
                continue

    return "❌ Sorry, sare free models abhi busy hain. Thodi der baad try karein ya Local Model use karein."

print("✅ Gemini ready (with multi-model fallback & rate-limit handling)")

✅ Gemini ready (with multi-model fallback & rate-limit handling)


In [ ]:
# ============================================================
# CELL 7b: OpenRouter Setup (Model: openrouter/owl-alpha)
# ============================================================
def setup_openrouter():
    api_key = "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"  # aapki key
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )
    return client

def openrouter_answer(client, context, question):
    prompt = f"""You are an AI assistant that answers questions strictly based on the provided context.
If the answer is not in the context, say 'Not found in the documents'.

Context:
{context}

Question: {question}"""
    response = client.chat.completions.create(
        model="openrouter/owl-alpha",   # ← aapka diya hua model
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    return response.choices[0].message.content

print("✅ OpenRouter ready (openrouter/owl-alpha)")

✅ OpenRouter ready (openrouter/owl-alpha)


In [ ]:
# ============================================================
# CELL 7c: Local Hugging Face Model (Flan-T5, fixed)
# ============================================================
# Ensure latest transformers and accelerate for compatibility
!pip install -q -U transformers accelerate

def setup_local_model():
    model_name = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if torch.cuda.is_available():
        model = model.to("cuda")
    return model, tokenizer

def local_answer(model, tokenizer, context, question):
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=None
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer
print("✅ Local model functions ready")

✅ Local model functions ready


In [ ]:
# ============================================================
# CELL 8: Document upload karwayein, phir LLM choose karwayein
# ============================================================
from google.colab import files

# Clear previous uploads to ensure fresh state
if os.path.exists('/content/uploaded_doc'):
    shutil.rmtree('/content/uploaded_doc')
os.makedirs('/content/uploaded_doc', exist_ok=True)

upload_path_base = "/content/uploaded_doc/"
print("📤 Document file upload karein (.pdf, .txt, .docx, .csv supported):")
uploaded = files.upload()

num_chunks = 0
for fn in uploaded.keys():
    full_upload_path = os.path.join(upload_path_base, fn)
    shutil.move(fn, full_upload_path)
    print(f"Uploaded file: {fn}")
    # Document ingest
    num_chunks = ingest_document(full_upload_path)
    break # Only process the first uploaded file for simplicity in this example

if num_chunks > 0:
    print(f"✅ Document processed: {num_chunks} chunks index ho gaye.\n")
else:
    print("❌ Document processing failed or no chunks were indexed.\n")

# LLM selection
print("🔍 LLM Select Karein:")
print("1. Gemini (Free Tier)")
print("2. OpenRouter (Free Model)")
print("3. Local Model (Flan-T5)")

choice = input("Apna choice number likhein (1/2/3): ").strip()

if choice == '1':
    llm_model = setup_gemini()
    answer_fn = lambda ctx, q: gemini_answer(llm_model, ctx, q)
    model_name = "Gemini"
elif choice == '2':
    openrouter_client = setup_openrouter()
    answer_fn = lambda ctx, q: openrouter_answer(openrouter_client, ctx, q)
    model_name = "OpenRouter"
elif choice == '3':
    local_model_flan, local_tokenizer_flan = setup_local_model()
    answer_fn = lambda ctx, q: local_answer(local_model_flan, local_tokenizer_flan, ctx, q)
    model_name = "Local Model (Flan-T5)"
else:
    print("Galat choice! Default Local Model (Flan-T5) use hoga.")
    local_model_flan, local_tokenizer_flan = setup_local_model()
    answer_fn = lambda ctx, q: local_answer(local_model_flan, local_tokenizer_flan, ctx, q)
    model_name = "Local Model (Flan-T5)"

print(f"✅ {model_name} ready hai. Ab sawal pucho.\n")

📤 Document file upload karein (.pdf, .txt, .docx, .csv supported):


Saving Comprehensive_Quiz.pdf to Comprehensive_Quiz.pdf
Uploaded file: Comprehensive_Quiz.pdf
✅ Document processed: 17 chunks index ho gaye.

🔍 LLM Select Karein:
1. Gemini (Free Tier)
2. OpenRouter (Free Model)
3. Local Model (Flan-T5)
Apna choice number likhein (1/2/3): 3


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Local Model (Flan-T5) ready hai. Ab sawal pucho.



In [ ]:
# ============================================================
# CELL 9: Sawal poocho, jawab pao (Interactive)
# ============================================================
print("📝 Ab aap document ke baare mein sawal pooch sakte hain.")
print("Type 'exit' likhkar program band kar sakte hain.\n")

while True:
    query = input("❓ Aap ka sawal: ").strip()
    if query.lower() == 'exit':
        print("👋 Program khatam!")
        break

    retrieved_chunks = retrieve(query, top_k=3)
    context = "\n\n".join(retrieved_chunks)
    answer = answer_fn(context, query)

    print("\n📝 Jawab:")
    print(textwrap.fill(answer, width=100))
    print("\n🔍 Source Chunks (jinse jawab aaya):")
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"[{i}] {chunk[:200]}...")
    print("-" * 60)

📝 Ab aap document ke baare mein sawal pooch sakte hain.
Type 'exit' likhkar program band kar sakte hain.

❓ Aap ka sawal: What is the main topic of the document?

📝 Jawab:
ews articles

🔍 Source Chunks (jinse jawab aaya):
[1] ews articles  
Each record includes:  
 Title  
 Text (main content)  
 Subject  
 Date  
The datasets were merged and labeled:  
 Fake → 1  
 Real → 0  
 
3. Data Cleaning & Preprocessing  
3.1...
[2] eated:  
 Article length  
 Word count  
 Average word length  
 Number of uppercase words  
 
4. Exploratory Dat a Analysis (EDA)  
4.1 Class Distribution  
The dataset shows a balanced/imbalance...
[3]  fake and real news articles using Machine Learning and Neural 
Networks. The system processes textual data, extracts meaningful features using TF -IDF, and applies 
multiple classification models. A ...
------------------------------------------------------------
❓ Aap ka sawal: exit
👋 Program khatam!


In [ ]:
# ============================================================
# CELL 10: System evaluation (manual)
# ============================================================
print("🧪 EVALUATION: Pehle se tayyar sawalon par system test karein.\n")

test_questions = [
    "What is the main topic of the document?",
    "What are the key points?",
    "Summarize the document in one sentence."
]

for q in test_questions:
    chunks = retrieve(q, top_k=3)
    ctx = "\n\n".join(chunks)
    ans = answer_fn(ctx, q)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print(f"Retrieved chunks: {len(chunks)}")
    print("-" * 50)

🧪 EVALUATION: Pehle se tayyar sawalon par system test karein.

Q: What is the main topic of the document?
A: ews articles...
Retrieved chunks: 3
--------------------------------------------------
Q: What are the key points?
A: Fake News Detection using Machine Learning & Neural Networks...
Retrieved chunks: 3
--------------------------------------------------
Q: Summarize the document in one sentence.
A: ews articles Each record includes:  Title  Text (main content)  Subject  Date The datasets were merged and labeled:  Fake  1  Real  0 3. Data Cleaning & Preprocessing 3.1 Data Cleaning Steps  Removed HTML tags  Converted text to lowercase  Removed punctuation and numbers  Removed extra whitespace  D...
Retrieved chunks: 3
--------------------------------------------------
